# 06 · Gold publication and reverse ETL

The last notebook of the pipeline. It closes the loop opened in 01: the data left
Lakebase, went through bronze, silver and a model, and now comes back as
predictions the retention team can act on.

Three things happen here.

**Scoring the whole portfolio.** All 10,000 customers, not just the test set. In
production you score everyone; the test set was for measuring, not for operating.

**Reverse ETL.** The predictions travel back to Postgres, where the CRM reads them.
This is the arrow that makes the architecture worth building — without it the work
sits in a warehouse nobody opens on a Tuesday morning.

**The public snapshot.** A parquet export of gold that the Streamlit app reads. The
app deployed on Databricks queries the warehouse; the public copy reads this file,
so anyone can open it without an account.

In [2]:
import sys
sys.path.append("../src")

from config import bootstrap

ctx = bootstrap()
spark, w = ctx.spark, ctx.w

connected to Databricks
  branch   : sandbox
  catalog  : bank_churn_eng
  identity : resolved (19 chars, not printed)


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


In [3]:
import io, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow

from config import UC_CATALOG, PG_SCHEMA, CAPACITY, VALUE_TP, RANDOM_STATE

pd.set_option("display.width", 175)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
CHURN, SAFE, GREY = "#a8471f", "#2c5a72", "#78736a"
SEED = RANDOM_STATE

MODEL_NAME    = "bank_churn_rf_calibrated"
MODEL_VERSION = "v1"

## 1 · Load the production model

The base reference is left by notebook 05. **This notebook is the only one that
writes `gold.production_model`**, the table the app reads.

In [4]:
prod = spark.table(f"{UC_CATALOG}.gold.production_model_base").toPandas().iloc[0]
met  = spark.table(f"{UC_CATALOG}.gold.model_metrics").toPandas().iloc[0]

print("production model")
print(f"  family          : {prod['family']}")
print(f"  gender excluded : {prod['gender_excluded']}")
print(f"  calibration     : {prod['calibration']}")
print(f"  ROC-AUC on test : {prod['test_roc_auc']:.4f}")
print(f"  Brier on test   : {prod['test_brier']:.4f}")

df = spark.table(f"{UC_CATALOG}.silver.customers_clean").toPandas()
df["exited"] = df["exited"].astype(bool)

BINARY  = ["balance_zero", "is_active_member", "has_cr_card"]
EXCLUDE = ["customer_id", "credit_score_band"]
FEATURES = [c for c in df.columns if c not in EXCLUDE + ["exited"] and not c.startswith("_")]

X = df[FEATURES].copy()
for c in BINARY:
    X[c] = X[c].astype(int)

# Load from MLflow when the run exists. When it does not -- notebook 05 stores an
# empty run_id if its logging step failed -- rebuild the same object instead of
# stopping here. With a fixed seed the rebuild is deterministic, so it is the
# same model and not an approximation of it.
model = None
if prod["run_id"]:
    try:
        model = mlflow.sklearn.load_model(f"runs:/{prod['run_id']}/model")
        print("\nmodel loaded from MLflow")
    except Exception as e:
        print(f"\n[warn] could not load from MLflow -- {type(e).__name__}: {e}")

if model is None:
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder

    print("\nrebuilding the production model from the stored hyperparameters...")
    ref = spark.table(f"{UC_CATALOG}.gold.model_reference").toPandas().iloc[0]
    params = json.loads(ref["params"])

    X_train, _, y_train, _ = train_test_split(
        X, df["exited"], test_size=0.20, stratify=df["exited"], random_state=SEED)

    CAT_PROD = ["geography"]                     # no gender, as decided in notebook 05
    NUM_PROD = ["credit_score", "balance", "tenure", "estimated_salary",
                "age", "num_of_products"]

    prep_prod = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), CAT_PROD),
        ("num", "passthrough", NUM_PROD),
        ("bin", "passthrough", BINARY)], remainder="drop", verbose_feature_names_out=False)

    base_prod = Pipeline([("prep", prep_prod),
                          ("clf", RandomForestClassifier(class_weight="balanced_subsample",
                                                         random_state=SEED, n_jobs=-1, **params))])
    model = CalibratedClassifierCV(base_prod, method="isotonic", cv=5)
    model.fit(X_train, y_train)
    print("model rebuilt")

print(f"\n{len(df):,} customers to score")

production model
  family          : random_forest
  gender excluded : True
  calibration     : isotonic_cv5
  ROC-AUC on test : 0.8562
  Brier on test   : 0.1028



model loaded from MLflow

10,000 customers to score


## 2 · Score the portfolio

In [5]:
df["churn_probability"] = model.predict_proba(X)[:, 1]

expected = df["churn_probability"].sum()
actual   = int(df["exited"].sum())

print(df["churn_probability"].describe().round(4).to_string())
print(f"\nexpected churners (sum of probabilities): {expected:,.0f}")
print(f"actual churners in the data             : {actual:,}")
print(f"deviation: {abs(expected - actual) / actual:.1%}")
print("\nThose two numbers agreeing is a consequence of calibration. An")
print("uncalibrated model would overstate the total.")

count    10000.0000
mean         0.2074
std          0.2533
min          0.0000
25%          0.0389
50%          0.0892
75%          0.2743
max          0.9907

expected churners (sum of probabilities): 2,074
actual churners in the data             : 2,037
deviation: 1.8%

Those two numbers agreeing is a consequence of calibration. An
uncalibrated model would overstate the total.


## 3 · Risk levels

The pre-registered method from notebook 00, with the quota split across countries
in proportion to **expected** churners rather than a single global threshold. That
is the strategy notebook 05 priced.

In [6]:
expected_by_country = df.groupby("geography")["churn_probability"].sum()
quota = (expected_by_country / expected_by_country.sum() * CAPACITY).round().astype(int)
quota.iloc[quota.values.argmax()] += CAPACITY - quota.sum()   # rounding adjustment

split = pd.DataFrame({
    "customers":        df.groupby("geography").size(),
    "expected_churn":   expected_by_country.round(0),
    "contact_quota":    quota,
})
print(split.to_string())
print(f"\ntotal: {quota.sum()} contacts")

           customers  expected_churn  contact_quota
geography                                          
France          5014           828.0            319
Germany         2509           834.0            322
Spain           2477           412.0            159

total: 800 contacts


In [7]:
# Tie-breaking matters here. Isotonic calibration is a step function, so many
# customers land on exactly the same probability. Among ties the INACTIVE one
# goes first, because that is who the bank can act on. customer_id settles any
# remaining tie and makes the result reproducible.
df["risk_level"] = "low"

for country, k in quota.items():
    sub = (df[df["geography"] == country]
           .sort_values(["churn_probability", "is_active_member", "customer_id"],
                        ascending=[False, True, True]))
    df.loc[sub.index[:k], "risk_level"] = "high"

from config import BREAK_EVEN
is_medium = (df["risk_level"] == "low") & (df["churn_probability"] >= BREAK_EVEN)
df.loc[is_medium, "risk_level"] = "medium"

df["predicted_class"] = df["risk_level"] == "high"

print(df["risk_level"].value_counts().reindex(["high", "medium", "low"]).to_string())
print("\ncalibration check by level -- these two columns should track each other:")
print(df.groupby("risk_level").agg(
    customers=("exited", "size"),
    mean_predicted=("churn_probability", lambda s: round(s.mean() * 100, 1)),
    actual_churn=("exited", lambda s: round(s.mean() * 100, 1)),
).reindex(["high", "medium", "low"]).to_string())

risk_level
high       800
medium    2431
low       6769

calibration check by level -- these two columns should track each other:
            customers  mean_predicted  actual_churn
risk_level                                         
high              800            87.4          93.4
medium           2431            38.0          42.1
low              6769             6.6           3.9


In [8]:
# Suggested action, anchored to the EDA findings. The only actionable variable is
# is_active_member; num_of_products >= 3 marks the segment that churns above 85%.
conditions = [
    (df.risk_level == "high") & (~df.is_active_member),
    (df.risk_level == "high") & (df.num_of_products >= 3),
    (df.risk_level == "high"),
    (df.risk_level == "medium"),
]
actions = [
    "Priority reactivation: phone contact with a usage incentive",
    "Portfolio review: audit the combination of products held",
    "Loyalty: commercial contact and review of terms",
    "Watch: include if budget remains after the high-risk group",
]
df["suggested_action"] = np.select(conditions, actions, default="No action")

print(df[df.risk_level != "low"]["suggested_action"].value_counts().to_string())

suggested_action
Watch: include if budget remains after the high-risk group     2431
Priority reactivation: phone contact with a usage incentive     654
Portfolio review: audit the combination of products held        105
Loyalty: commercial contact and review of terms                  41


## 4 · Gold tables

In [9]:
gold = df[["customer_id", "churn_probability", "predicted_class", "risk_level",
           "suggested_action", "geography", "gender", "age_group", "products_group",
           "is_active_member", "balance_zero", "exited"]].copy()
gold = gold.rename(columns={"exited": "actual_churn"})
gold["model_name"]    = MODEL_NAME
gold["model_version"] = MODEL_VERSION
gold["scored_at"]     = pd.Timestamp.utcnow()

(spark.createDataFrame(gold).write.format("delta").mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(f"{UC_CATALOG}.gold.customer_churn_predictions"))

spark.sql(f"""
    COMMENT ON TABLE {UC_CATALOG}.gold.customer_churn_predictions IS
    'Gold. Full portfolio scored by the calibrated production model, with risk
     level and suggested action. Source for the reverse ETL and the application.'
""")
print("written: gold.customer_churn_predictions")

written: gold.customer_churn_predictions


## 5 · Reverse ETL to Lakebase

This is where the database stops being decoration. Two writes, both idempotent:
the run is upserted, and the predictions are deleted for that run before being
reloaded. Running the cell twice leaves the same state as running it once.

In [10]:
notes = (f"RF without gender, isotonic calibration. Per-country quota split. "
         f"Test ROC-AUC {prod['test_roc_auc']:.4f}, Brier {prod['test_brier']:.4f}.")

with ctx.connect() as conn:
    cur = conn.cursor()
    cur.execute(f"""
        INSERT INTO {PG_SCHEMA}.model_runs
            (model_name, model_version, decision_threshold, trained_at, notes)
        VALUES (%s, %s, %s, now(), %s)
        ON CONFLICT (model_name, model_version)
        DO UPDATE SET decision_threshold = EXCLUDED.decision_threshold,
                      notes = EXCLUDED.notes
        RETURNING model_run_id;
    """, (MODEL_NAME, MODEL_VERSION, float(prod["threshold"]), notes))
    MODEL_RUN_ID = cur.fetchone()[0]
    conn.commit()

print("model_run_id:", MODEL_RUN_ID)

model_run_id: 1


In [11]:
payload = pd.DataFrame({
    "customer_id":       df["customer_id"].astype("int64"),
    "model_run_id":      MODEL_RUN_ID,
    "churn_probability": df["churn_probability"].round(5),
    "predicted_class":   df["predicted_class"],
    "risk_level":        df["risk_level"],
})

buf = io.StringIO()
payload.to_csv(buf, index=False, header=False)
buf.seek(0)

with ctx.connect() as conn:
    cur = conn.cursor()
    cur.execute(f"DELETE FROM {PG_SCHEMA}.customer_predictions WHERE model_run_id = %s",
                (MODEL_RUN_ID,))
    cur.execute(
        f"COPY {PG_SCHEMA}.customer_predictions "
        f"(customer_id, model_run_id, churn_probability, predicted_class, risk_level) "
        f"FROM STDIN WITH (FORMAT CSV)", stream=buf)
    conn.commit()

n = ctx.query(f"SELECT COUNT(*) AS n FROM {PG_SCHEMA}.customer_predictions "
              f"WHERE model_run_id = %s", (MODEL_RUN_ID,)).iloc[0, 0]
print(f"predictions written to Lakebase: {n:,}")

predictions written to Lakebase: 10,000


In [12]:
print("-- what the campaigns team sees --")
print(ctx.query(f"""
    SELECT risk_level, COUNT(*) AS customers,
           ROUND(AVG(churn_probability)::numeric, 4) AS mean_probability
    FROM   {PG_SCHEMA}.customer_predictions
    WHERE  model_run_id = %s
    GROUP  BY risk_level ORDER BY mean_probability DESC
""", (MODEL_RUN_ID,)).to_string(index=False))

print("\n-- the first ten customers to call tomorrow --")
print(ctx.query(f"""
    SELECT p.customer_id,
           ROUND(p.churn_probability::numeric, 4) AS probability,
           p.risk_level, g.country_name AS country,
           c.age, c.is_active_member AS active, c.num_of_products AS products
    FROM   {PG_SCHEMA}.customer_predictions p
    JOIN   {PG_SCHEMA}.customers   c USING (customer_id)
    JOIN   {PG_SCHEMA}.geographies g ON g.geography_id = c.geography_id
    WHERE  p.model_run_id = %s AND p.risk_level = 'high'
    ORDER  BY p.churn_probability DESC
    LIMIT  10
""", (MODEL_RUN_ID,)).to_string(index=False))

print("\nNo surname anywhere. The privacy boundary set in notebook 01 holds all")
print("the way to the operational query.")

-- what the campaigns team sees --
risk_level  customers mean_probability
      high        800           0.8745
    medium       2431           0.3802
       low       6769           0.0665

-- the first ten customers to call tomorrow --
 customer_id probability risk_level country  age  active  products
    15731815      0.9907       high   Spain   63   False         3
    15662291      0.9907       high  France   55   False         3
    15805637      0.9907       high  France   36   False         3
    15606613      0.9907       high  France   59   False         1
    15588614      0.9907       high  France   57   False         1
    15640846      0.9907       high Germany   58   False         4
    15579212      0.9907       high  France   57   False         1
    15651823      0.9907       high  France   60   False         1
    15689886      0.9907       high Germany   39    True         3
    15609623      0.9907       high  France   63   False         1

No surname anywhere. Th

## 6 · Supporting tables for the application

The model comparison, the confusion matrix and the feature importances the app
displays come from here.

In [13]:
comparison = None
try:
    user = spark.sql("SELECT current_user()").first()[0]
    runs = mlflow.search_runs(experiment_names=[f"/Users/{user}/bank_churn_eng"])
    metric_cols = [c for c in runs.columns
                   if c.startswith("metrics.cv_") and not c.endswith("_std")]
    comparison = (runs[["tags.mlflow.runName"] + metric_cols]
                  .rename(columns={"tags.mlflow.runName": "model"})
                  .rename(columns={c: c.replace("metrics.cv_", "") for c in metric_cols})
                  .dropna(subset=["average_precision"]))

    # The experiment accumulates one run per execution, so re-running notebook 04
    # leaves several runs with the same name. Without this the app stacks the bars
    # and they measure more than their labels say.
    # final_model is the same estimator as the winning *_tuned run, logged again
    # under a production name with only two metrics. Keeping it would draw a
    # second identical bar with most columns empty.
    comparison = comparison[comparison["model"] != "final_model"]

    comparison = (comparison.sort_values("average_precision")
                            .drop_duplicates(subset="model", keep="last")
                            .sort_values("average_precision", ascending=False))
    comparison["is_baseline"] = comparison["model"].str.contains("baseline")

    (spark.createDataFrame(comparison).write.format("delta").mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable(f"{UC_CATALOG}.gold.model_comparison"))
    print(comparison.round(4).to_string(index=False))
except Exception as e:
    print(f"[warn] could not build model_comparison ({type(e).__name__})")

                     model  recall  average_precision     f1  precision  roc_auc  accuracy  is_baseline
       random_forest_tuned  0.6564             0.6848 0.6110     0.5721   0.8593    0.8301        False
random_forest_no_sensitive  0.6258             0.6552 0.5881     0.5553   0.8420    0.8216        False
            logistic_tuned  0.7442             0.6454 0.5741     0.4675   0.8416    0.7751        False
                tree_tuned  0.7276             0.6389 0.5753     0.4763   0.8325    0.7811        False
         baseline_business  0.2006             0.4677 0.3216     0.8165   0.7526    0.8280         True
            baseline_dummy  0.0000             0.2037 0.0000     0.0000   0.5000    0.7962         True


In [14]:
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

y = df["exited"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)

# The production model is wrapped by CalibratedClassifierCV, so the preprocessor
# sits one level deeper than in notebook 04.
prep_fit = (model.calibrated_classifiers_[0].estimator.named_steps["prep"]
            if hasattr(model, "calibrated_classifiers_")
            else model.named_steps["prep"])
# transformers_ also carries the ('remainder', 'drop', [...]) entry. Including
# it would list precisely the columns the pipeline THROWS AWAY, which is trap 1
# happening for real: they come out with importance exactly zero and push the
# variables that matter down the table.
used = sorted({c for name, trans, cols in prep_fit.transformers_
               if name != "remainder" and trans != "drop"
               for c in (cols if isinstance(cols, list) else [])})

perm = permutation_importance(model, X_te[used], y_te, scoring="average_precision",
                              n_repeats=10, random_state=SEED, n_jobs=-1)

fi = pd.DataFrame({"feature": used,
                   "importance": perm.importances_mean,
                   "std": perm.importances_std})
fi["distinguishable"] = fi["importance"] > fi["std"]
fi = fi.sort_values("importance", ascending=False)

(spark.createDataFrame(fi).write.format("delta").mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(f"{UC_CATALOG}.gold.feature_importance"))
print(fi.round(4).to_string(index=False))

         feature  importance    std  distinguishable
             age      0.2632 0.0168             True
 num_of_products      0.2613 0.0138             True
is_active_member      0.1035 0.0106             True
       geography      0.0511 0.0065             True
         balance      0.0456 0.0078             True
          tenure      0.0042 0.0021             True
    balance_zero      0.0025 0.0026            False
    credit_score      0.0012 0.0015            False
estimated_salary      0.0006 0.0036            False
     has_cr_card      0.0003 0.0011            False


## 7 · Register the model in Unity Catalog

The application cannot load a model through `runs:/<run_id>`: MLflow experiments
have their own access control lists, separate from the catalog's, and a service
principal without permission gets **"Run not found"** — a message that hides a
permissions problem behind a not-found error.

Registering in Unity Catalog puts the model under the same permission system as
the tables, which is the actual fix.

This cell is the **only** writer of `gold.production_model`.

In [15]:
from mlflow import MlflowClient
from mlflow.models import infer_signature

UC_MODEL = f"{UC_CATALOG}.gold.churn_model"
mlflow.set_registry_uri("databricks-uc")

client = MlflowClient()

# Reuse the registered version if this run was already signed. Without this guard
# every re-execution creates a new version of an identical model.
previous = [v for v in client.search_model_versions(f"name='{UC_MODEL}'")
            if isinstance(getattr(v, "tags", None), dict)
            and v.tags.get("source_run") == prod["run_id"]]

if previous:
    UC_VERSION  = max(int(v.version) for v in previous)
    SIGNED_RUN  = client.get_model_version(UC_MODEL, str(UC_VERSION)).run_id
    print(f"existing version reused: {UC_MODEL} v{UC_VERSION}")
else:
    # Integer columns in a signature are a trap: Python integers cannot hold a
    # missing value, so if the application ever sends a NaN the column arrives
    # as float and schema enforcement rejects the request. Declaring them as
    # doubles up front is what MLflow warns about, and it costs nothing.
    example = X[used].head(5).copy()
    for _c in example.select_dtypes(include="integer").columns:
        example[_c] = example[_c].astype("float64")

    signature = infer_signature(example, model.predict_proba(example)[:, 1])

    # MLflow serialises sklearn models with skops, which refuses to rebuild
    # classes outside its allowlist -- a real protection, because a pickle can
    # execute arbitrary code when loaded. CalibratedClassifierCV holds a list of
    # private _CalibratedClassifier objects, so it has to be declared trusted.
    TRUSTED = ["sklearn.calibration._CalibratedClassifier"]

    with mlflow.start_run(run_name="production_model_signed") as run:
        SIGNED_RUN = run.info.run_id          # captured first, so a later failure
                                              # does not lose a run that exists
        try:
            info = mlflow.sklearn.log_model(model, name="model", signature=signature,
                                            input_example=example,
                                            registered_model_name=UC_MODEL,
                                            skops_trusted_types=TRUSTED)
        except TypeError:
            # Older MLflow has no skops path; pickle is the only format there.
            info = mlflow.sklearn.log_model(model, name="model", signature=signature,
                                            input_example=example,
                                            registered_model_name=UC_MODEL)
    UC_VERSION = info.registered_model_version
    client.set_model_version_tag(UC_MODEL, str(UC_VERSION), "source_run", prod["run_id"])
    print(f"registered: {UC_MODEL} v{UC_VERSION}")

link = pd.DataFrame([{**prod.to_dict(),
                      "uc_model": UC_MODEL,
                      "uc_version": str(UC_VERSION),
                      "signed_run": SIGNED_RUN}])

(spark.createDataFrame(link).write.format("delta").mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(f"{UC_CATALOG}.gold.production_model"))
print("written: gold.production_model")

🔗 View Logged Model at: https://dbc-04c9ea21-2a95.cloud.databricks.com/ml/experiments/733594851517913/models/m-<run_id>


Registered model 'bank_churn_eng.gold.churn_model' already exists. Creating a new version of this model...


🔗 Created version '4' of model 'bank_churn_eng.gold.churn_model': https://dbc-04c9ea21-2a95.cloud.databricks.com/explore/data/models/bank_churn_eng/gold/churn_model/version/4


🏃 View run production_model_signed at: https://dbc-04c9ea21-2a95.cloud.databricks.com/ml/experiments/733594851517913/runs/<run_id>
🧪 View experiment at: https://dbc-04c9ea21-2a95.cloud.databricks.com/ml/experiments/733594851517913
registered: bank_churn_eng.gold.churn_model v4
written: gold.production_model


## 8 · The public snapshot

Everything above runs on Databricks. This cell produces the one artefact that
leaves it: a parquet export of the gold tables, committed to the repository, which
the Streamlit Community Cloud app reads.

**Why not query the warehouse from the public app.** It would need credentials
stored on a third-party service, and a Free Edition SQL warehouse stops itself
after inactivity — so the app would be down more often than up. The snapshot costs
a few hundred kilobytes and works for anyone with a browser.

The pipeline still proves the Databricks work. A dashboard does not need a query
engine behind it to display ten thousand rows.

In [16]:
from pathlib import Path

OUT = Path("../data")
OUT.mkdir(exist_ok=True)

exports = {
    "predictions":       gold,
    "model_metrics":     spark.table(f"{UC_CATALOG}.gold.model_metrics").toPandas(),
    "feature_importance": fi,
}
if comparison is not None:
    exports["model_comparison"] = comparison

manifest = []
for name, frame in exports.items():
    # toPandas() on Spark Connect attaches query metrics to df.attrs, and pandas
    # serialises attrs to JSON when writing parquet. PlanMetrics is not
    # serialisable, so the write fails on a detail that has nothing to do with
    # the data. Dropping attrs on a copy is enough, and leaves the original
    # DataFrame untouched for the cells below.
    frame = frame.copy()
    frame.attrs = {}

    path = OUT / f"{name}.parquet"
    frame.to_parquet(path, index=False, compression="snappy")
    manifest.append({"file": path.name, "rows": len(frame),
                     "kb": round(path.stat().st_size / 1024, 1)})

pd.DataFrame([{
    "generated_at": pd.Timestamp.utcnow().isoformat(),
    "model_name": MODEL_NAME, "model_version": MODEL_VERSION,
    "uc_model": UC_MODEL, "uc_version": str(UC_VERSION),
    "capacity": CAPACITY, "threshold": float(prod["threshold"]),
    "test_roc_auc": float(prod["test_roc_auc"]),
    "test_brier": float(prod["test_brier"]),
}]).to_json(OUT / "manifest.json", orient="records", indent=2)

print(pd.DataFrame(manifest).to_string(index=False))
print(f"\ntotal: {sum(m['kb'] for m in manifest):.1f} KB -- small enough to commit")

                      file  rows    kb
       predictions.parquet 10000 131.1
     model_metrics.parquet     1  12.2
feature_importance.parquet    10   3.0
  model_comparison.parquet     6   5.3

total: 151.6 KB -- small enough to commit


## 9 · Prescriptive analysis

Three actions for the high-risk segment, each anchored to a finding rather than to
intuition, and each with the value at stake if it works.

In [17]:
seg = df[df.risk_level == "high"].copy()

recs = []
for mask, name in [
    (~seg.is_active_member,        "1. Reactivate inactive customers"),
    (seg.num_of_products >= 3,     "2. Audit portfolios with 3+ products"),
    (seg.age_group.isin(["50-59", "60+"]), "3. Retain the over-50 segment"),
]:
    recs.append({"action": name,
                 "customers": int(mask.sum()),
                 "expected_churners": round(seg.loc[mask, "churn_probability"].sum()),
                 "value_if_successful_eur": round(seg.loc[mask, "churn_probability"].sum() * VALUE_TP)})

print(pd.DataFrame(recs).to_string(index=False))
print(f"\ntotal expected churners in the high-risk segment: "
      f"{seg['churn_probability'].sum():,.0f}")
print("\nThe groups overlap on purpose: a customer can be both inactive and over")
print("50. They are lenses on the same segment, not a partition of it.")

                              action  customers  expected_churners  value_if_successful_eur
    1. Reactivate inactive customers        654                574                    83195
2. Audit portfolios with 3+ products        264                239                    34693
       3. Retain the over-50 segment        438                392                    56842

total expected churners in the high-risk segment: 700

The groups overlap on purpose: a customer can be both inactive and over
50. They are lenses on the same segment, not a partition of it.


## 10 · End-to-end verification

In [18]:
checks = []

n_pg = int(ctx.query(f"SELECT COUNT(*) AS n FROM {PG_SCHEMA}.customer_predictions "
                     f"WHERE model_run_id = %s", (MODEL_RUN_ID,)).iloc[0, 0])
n_gold = spark.table(f"{UC_CATALOG}.gold.customer_churn_predictions").count()
checks.append(("Gold and Lakebase hold the same rows",
               n_gold == n_pg == len(df), f"{n_gold} / {n_pg}"))

orphans = int(ctx.query(f"""
    SELECT COUNT(*) AS n FROM {PG_SCHEMA}.customer_predictions p
    LEFT JOIN {PG_SCHEMA}.customers c USING (customer_id)
    WHERE c.customer_id IS NULL""").iloc[0, 0])
checks.append(("Referential integrity", orphans == 0, f"{orphans} orphans"))

n_high = int((df.risk_level == "high").sum())
checks.append(("High-risk equals capacity", n_high == CAPACITY, f"{n_high} / {CAPACITY}"))

checks.append(("Snapshot exported for the app",
               (OUT / "predictions.parquet").exists(), "data/predictions.parquet"))

checks.append(("No surname anywhere downstream",
               "surname" not in gold.columns, "gold has no surname column"))

for name, ok, evidence in checks:
    print(f"  {'PASS' if ok else 'FAIL'} {name:<42} {evidence}")

print(f"\n{sum(1 for _, ok, _ in checks if ok)} of {len(checks)} passed")

  PASS Gold and Lakebase hold the same rows       10000 / 10000
  PASS Referential integrity                      0 orphans
  PASS High-risk equals capacity                  800 / 800
  PASS Snapshot exported for the app              data/predictions.parquet
  PASS No surname anywhere downstream             gold has no surname column

5 of 5 passed


In [19]:
import joblib

path = OUT / "model.joblib"
joblib.dump(model, path, compress=3)
size_mb = path.stat().st_size / 1024**2

print(f"model.joblib: {size_mb:.1f} MB")
if size_mb > 50:
    print("  Too large to commit comfortably. GitHub warns above 50 MB and blocks")
    print("  at 100 MB. Either leave the tab disabled or use git-lfs.")
    path.unlink()
    print("  removed -- the app will explain the absence instead")
else:
    print("  small enough to commit: the individual prediction tab will work")

model.joblib: 26.8 MB
  small enough to commit: the individual prediction tab will work


---

## Summary

The pipeline is closed:

```
Volume ─▶ Lakebase ─▶ bronze ─▶ silver ─▶ model ─▶ gold ─▶ Lakebase ─▶ CRM
                                                      └──▶ data/*.parquet ─▶ public app
```

| | |
|---|---|
| Customers scored | 10,000 |
| High risk | 800, split across countries by expected churners |
| Written back to Lakebase | `customer_predictions`, append-only history |
| Registered in Unity Catalog | `gold.churn_model`, with signature |
| Public snapshot | `data/*.parquet`, a few hundred KB |

**Next:** the Streamlit app in `app/`, which reads the snapshot.